# Camada Gold — Pronta para Análise

A camada **Gold** recebe os dados limpos da **Silver** e cria **tabelas analíticas** prontas para consumo de negócio:

1. **sales_enriched** — Fatos de venda enriquecidos com dados do cliente e do produto (tabela denormalizada)
2. **customer_summary** — Métricas agregadas por cliente (total gasto, nº de pedidos, ticket médio, etc.)
3. **product_summary** — Métricas agregadas por produto (receita total, unidades vendidas, avaliação média, etc.)

> Pense na Gold como o **prato pronto** na mesa — os ingredientes já foram comprados (Bronze), lavados (Silver), e agora estão preparados e servidos para consumo.

In [0]:
%sql
-- ============================================================
-- GOLD 1: sales_enriched
-- ============================================================
-- Esta tabela faz o JOIN das 3 tabelas Silver para criar uma
-- tabela fata denormalizada: cada linha = 1 venda com dados do
-- cliente e do produto ja incluidos.
-- NAO ha agregacao — e uma visao " Wide " (larga) de cada venda.

CREATE OR REPLACE TABLE capgemini_trainingforthecase.gold.sales_enriched AS
SELECT
    -- Dados do pedido
    s.order_id,
    s.order_date,
    s.order_time,
    s.delivery_date,
    s.quantity,
    s.unit_price,
    s.order_value,
    s.shipping_cost,
    s.coupon_code,
    s.coupon_discount,
    s.total_amount,
    s.payment_mode,
    s.order_status,
    s.rating,
    s.review_text,

    -- Dados do cliente (da Silver.customers)
    c.customer_name,
    c.gender,
    c.age,
    c.age_group,
    c.city        AS customer_city,
    c.state       AS customer_state,
    c.customer_tier,

    -- Dados do produto (da Silver.products)
    p.product_name,
    p.category,
    p.brand,
    p.original_price,
    p.discount_percent,
    p.selling_price,

    -- Metrica calculada: tempo de entrega em dias
    datediff(s.delivery_date, s.order_date) AS delivery_days
FROM capgemini_trainingforthecase.silver.sales      AS s
JOIN capgemini_trainingforthecase.silver.customers  AS c
  ON s.customer_id = c.customer_id
JOIN capgemini_trainingforthecase.silver.products   AS p
  ON s.product_id  = p.product_id;

In [0]:
%sql
-- ============================================================
-- GOLD 2: customer_summary
-- ============================================================
-- Agrega as vendas por cliente, calculando metricas de negocio:
--   - Total gasto, numero de pedidos, ticket medio
--   - Categoria favorita e forma de pagamento preferida
--   - Avaliacao media dada pelo cliente

CREATE OR REPLACE TABLE capgemini_trainingforthecase.gold.customer_summary AS

WITH base AS (
    SELECT
        s.customer_id,
        c.customer_name,
        c.customer_tier,
        c.city,
        c.state,
        s.order_id,
        s.total_amount,
        s.product_id,
        s.order_date,
        s.payment_mode,
        p.category,
        s.rating
    FROM capgemini_trainingforthecase.silver.sales    AS s
    JOIN capgemini_trainingforthecase.silver.customers AS c
      ON s.customer_id = c.customer_id
    JOIN capgemini_trainingforthecase.silver.products  AS p
      ON s.product_id  = p.product_id
),

metricas_gerais AS (
    SELECT
        customer_id,
        customer_name,
        customer_tier,
        city,
        state,
        COUNT(order_id)                     AS total_orders,
        ROUND(SUM(total_amount), 2)         AS total_spent,
        ROUND(AVG(total_amount), 2)         AS avg_order_value,
        MAX(total_amount)                   AS highest_order_value,
        COUNT(DISTINCT product_id)          AS unique_products_bought,
        COUNT(DISTINCT order_date)          AS active_days,
        ROUND(AVG(rating), 2)               AS avg_rating_given
    FROM base
    GROUP BY customer_id, customer_name, customer_tier, city, state
),

conta_categoria AS (
    -- Conta quantas vezes cada cliente comprou em cada categoria
    SELECT customer_id, category, COUNT(*) AS cnt_cat
    FROM base
    GROUP BY customer_id, category
),

conta_pagamento AS (
    -- Conta quantas vezes cada cliente usou cada forma de pagamento
    SELECT customer_id, payment_mode, COUNT(*) AS cnt_pay
    FROM base
    GROUP BY customer_id, payment_mode
)

SELECT
    mg.customer_id,
    mg.customer_name,
    mg.customer_tier,
    mg.city,
    mg.state,
    mg.total_orders,
    mg.total_spent,
    mg.avg_order_value,
    mg.highest_order_value,
    mg.unique_products_bought,
    mg.active_days,
    MAX_BY(cc.category, cc.cnt_cat)    AS favorite_category,
    MAX_BY(cp.payment_mode, cp.cnt_pay) AS preferred_payment_mode,
    mg.avg_rating_given
FROM metricas_gerais  AS mg
JOIN conta_categoria  AS cc  ON mg.customer_id = cc.customer_id
JOIN conta_pagamento  AS cp  ON mg.customer_id = cp.customer_id
GROUP BY
    mg.customer_id,
    mg.customer_name,
    mg.customer_tier,
    mg.city,
    mg.state,
    mg.total_orders,
    mg.total_spent,
    mg.avg_order_value,
    mg.highest_order_value,
    mg.unique_products_bought,
    mg.active_days,
    mg.avg_rating_given;

In [0]:
%sql
-- ============================================================
-- GOLD 3: product_summary
-- ============================================================
-- Agrega as vendas por produto, calculando:
--   - Receita total e unidades vendidas
--   - Numero de clientes unicos que compraram
--   - Avaliacao media e ticket medio

CREATE OR REPLACE TABLE capgemini_trainingforthecase.gold.product_summary AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    p.selling_price,
    p.avg_rating,

    -- Metricas de vendas
    COUNT(s.order_id)                              AS total_orders,
    SUM(s.quantity)                                AS total_units_sold,
    ROUND(SUM(s.total_amount), 2)                  AS total_revenue,
    ROUND(AVG(s.total_amount), 2)                   AS avg_order_value,
    COUNT(DISTINCT s.customer_id)                  AS unique_customers,

    -- Avaliacao media real (calculada a partir das vendas)
    ROUND(AVG(s.rating), 2)                         AS avg_customer_rating

FROM capgemini_trainingforthecase.silver.sales    AS s
JOIN capgemini_trainingforthecase.silver.products  AS p
  ON s.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    p.selling_price,
    p.avg_rating;

In [0]:
# ============================================================
# VERIFICACAO: conferir se as tabelas Gold foram criadas corretamente
# ============================================================

gold_tables = [
    ("sales_enriched", "capgemini_trainingforthecase.gold.sales_enriched"),
    ("customer_summary", "capgemini_trainingforthecase.gold.customer_summary"),
    ("product_summary", "capgemini_trainingforthecase.gold.product_summary"),
]

for name, table in gold_tables:
    df = spark.table(table)
    count = df.count()
    print(f"{name}: {count} linhas")
    print(f"  Colunas: {df.columns}")
    print()

print("Camada Gold criada com sucesso!")

In [0]:
%sql
CREATE VIEW capgemini_trainingforthecase.gold.vw_top_customers AS
     SELECT customer_name, total_spent, total_orders
     FROM capgemini_trainingforthecase.gold.customer_summary
     ORDER BY total_spent DESC;